# **LLM Serving with Apigee**

<table align="left">
    <td style="text-align: center">
        <a href="https://colab.research.google.com/github/GoogleCloudPlatform/apigee-samples/blob/main/llm-intelligent-routing/llm_intelligent_routing_v1.ipynb">
          <img src="https://github.com/GoogleCloudPlatform/apigee-samples/blob/main/images/icon32.png?raw=true" alt="Google Colaboratory logo"><br> Open in Colab
        </a>
      </td>
      <td style="text-align: center">
        <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fapigee-samples%2Fmain%2Fllm-intelligent-routing%2Fllm_intelligent_routing_v1.ipynb">
          <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
        </a>
      </td>
      <td style="text-align: center">
        <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/apigee-samples/main/llm-intelligent-routing/llm_intelligent_routing_v1.ipynb">
          <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Open in Workbench
        </a>
      </td>
      <td style="text-align: center">
        <a href="https://github.com/GoogleCloudPlatform/apigee-samples/blob/main/llm-intelligent-routing/llm_intelligent_routing_v1.ipynb">
          <img src="https://github.com/GoogleCloudPlatform/apigee-samples/blob/main/images/github-mark.png?raw=true" width="30" alt="GitHub logo"><br> View on GitHub
        </a>
      </td>
</table>
<br />
<br />
<br />

# Intelligent Model Routing

Not every prompt needs your most capable model. This sample routes each request to a fast,
inexpensive model or a slower, more capable one based on the semantic complexity of the
prompt — decided inside Apigee, with no LLM call in the decision path.

The classifier is a 1-nearest-neighbor lookup against a Vertex AI Vector Search index
containing only **complex** exemplars. Close to one means complex. Far from all of them
means not. Flash is the default; Pro is the opt-in.

## How it works

1. Apigee extracts the last user message and normalizes it.
2. An exact-match KVM lookup can pin the model and short-circuit everything.
3. Otherwise Apigee embeds the prompt with `text-embedding-005`.
4. `findNeighbors` returns the single nearest complex exemplar.
5. Above the similarity threshold routes to the complex model; anything else routes to the
   simple model.

Every response carries `x-routing-selected-model`, `x-routing-reason`, and
`x-routing-distance`.

## Setup

Use the following GCP CloudShell tutorial. Follow the instructions to deploy the sample.

[![Open in Cloud Shell](https://gstatic.com/cloudssh/images/open-btn.svg)](https://ssh.cloud.google.com/cloudshell/open?cloudshell_git_repo=https://github.com/GoogleCloudPlatform/apigee-samples&cloudshell_git_branch=main&cloudshell_workspace=.&cloudshell_tutorial=llm-intelligent-routing/docs/cloudshell-tutorial.md)

## Test Sample

### Initialize notebook variables

Set these to the values printed at the end of `deploy-llm-intelligent-routing.sh`.

In [ ]:
PROJECT_ID = "<your-project-id>"  # @param {type:"string"}
REGION = "<your-region>"  # @param {type:"string"}
APIGEE_HOST = "<your-apigee-host>"  # @param {type:"string"}
APIKEY = "<your-api-key>"  # @param {type:"string"}

ENDPOINT = (
    f"https://{APIGEE_HOST}/v1/samples/llm-intelligent-routing"
    f"/v1/projects/{PROJECT_ID}/locations/{REGION}"
    f"/publishers/google/models/auto:generateContent"
)

print(ENDPOINT)

### A helper that sends a prompt and reports the routing decision

The literal model name `auto` in the URL is the caller's way of saying "gateway, you pick".

In [ ]:
import requests


def route(prompt):
    """Send a prompt through the proxy and return the routing decision."""
    response = requests.post(
        ENDPOINT,
        headers={"Content-Type": "application/json", "x-apikey": APIKEY},
        json={"contents": [{"role": "user", "parts": [{"text": prompt}]}]},
        timeout=120,
    )
    response.raise_for_status()
    return {
        "prompt": prompt,
        "model": response.headers.get("x-routing-selected-model"),
        "reason": response.headers.get("x-routing-reason"),
        "distance": response.headers.get("x-routing-distance"),
    }


def show(result):
    print(f"prompt:   {result['prompt'][:70]}")
    print(f"model:    {result['model']}")
    print(f"reason:   {result['reason']}")
    print(f"distance: {result['distance']}")

### Test 1 — a simple prompt

A short factual question is far from every complex exemplar, so it should route to the
simple model with reason `no_complex_match`.

In [ ]:
result = route("What is the capital of France?")
show(result)

assert result["reason"] == "no_complex_match", result
assert "flash" in result["model"], result
print("\nPASS - simple prompt routed to the fast model")

### Test 2 — a complex prompt

This prompt is semantically close to the `complex_code_architecture_02` exemplar, so it
should clear the threshold and route to the capable model.

In [ ]:
result = route(
    "We have a distributed monolith with tangled dependencies between services. "
    "Propose a decomposition strategy, the order to execute it in, and how to keep "
    "the system releasable throughout."
)
show(result)

assert result["reason"] == "complex_match", result
assert "pro" in result["model"], result
print("\nPASS - complex prompt routed to the capable model")

### Test 3 — an off-topic prompt

This is the case that motivates indexing only complex exemplars.

If the index held both simple and complex examples and the router read the label off the
nearest neighbor's ID, this prompt would be routed by whichever of the 18 examples happened
to be nearest — however far away. The decision would be arbitrary and the recorded distance
would never be consulted.

With a single-class index there is no label to read. The prompt is far from every complex
exemplar, so it falls to the simple model and the response says exactly why.

In [ ]:
result = route("What's the weather in Lagos today?")
show(result)

assert result["reason"] == "no_complex_match", result
assert "flash" in result["model"], result
print("\nPASS - off-topic prompt fell to the fast model for a stated reason")

### Test 4 — the KVM override

`config/env__envname__llm-intelligent-routing-overrides__kvmfile__0.json` pins
`"draft our incident postmortem for the march outage."` to the complex model. A KVM hit
short-circuits the embedding call entirely, so no distance is recorded.

Keys are matched after lowercasing and trimming, and matching is exact — punctuation
differences will not hit.

In [ ]:
result = route("Draft our incident postmortem for the March outage.")
show(result)

assert result["reason"] == "kvm_override", result
assert result["distance"] == "n/a", result
print("\nPASS - pinned prompt bypassed the classifier")

### Choosing a threshold

`ROUTING_MIN_SIMILARITY` defaults to `0.75`, which is a starting point rather than a
validated constant. Run a spread of prompts and look at the actual distances: a good
threshold sits in the gap between the distances your simple traffic produces and the
distances your complex traffic produces.

If the two clusters overlap, the fix is better exemplars in
`config/complex-examples.json`, not a different threshold.

In [ ]:
import pandas as pd

prompts = [
    "What is 12 times 8?",
    "Translate 'good morning' into Japanese.",
    "Summarize this paragraph in one sentence.",
    "What's the weather in Lagos today?",
    "Prove this scheduling algorithm terminates and derive its worst-case complexity.",
    "Compare building in-house against three vendors across cost, lock-in, and risk, "
    "and recommend one with the reasoning made explicit.",
    "Read these five contracts and compare their termination clauses and liability caps, "
    "flagging anywhere the terms conflict.",
]

rows = [route(p) for p in prompts]
frame = pd.DataFrame(rows)
frame["prompt"] = frame["prompt"].str.slice(0, 60)
frame

### Clean up

Run `./undeploy-llm-intelligent-routing.sh` from the sample directory to remove the Apigee
artifacts. It prints, but does not run, the commands to delete the Vector Search index and
endpoint — those take 20–30 minutes to recreate.